# Clean Kivy to APK Compilation Notebook
This notebook provides a clean, modern build flow for compiling your Kivy / KivyMD application into an APK using Buildozer. All legacy `android-clang-wrapper` workarounds, environment overrides, and forced old Cython patches have been removed.

## Step 1: Clean Up Workspace & Remove Legacy Configurations
This removes any remnants of the old configurations, environment wrappers, and corrupted caches.

In [ ]:
!rm -f /content/android-clang-wrapper.sh
!rm -rf ~/.buildozer/
!rm -rf .pytest_cache/
!rm -rf Open_E-Dope sample_data bin main.py buildozer.spec layout.kv extra_manifest.xml extra_intents.xml assets app res circularprogressbar.py

## Step 2: Unset Legacy Environment Variables
This explicitly purges any forced compiler variables or pip constraint paths that cause the Android build system to crash.

In [ ]:
import os
vars_to_clear = ['CC', 'CXX', 'CPATH', 'C_INCLUDE_PATH', 'CPLUS_INCLUDE_PATH', 'CFLAGS', 'CPPFLAGS', 'CXXFLAGS', 'PKG_CONFIG_PATH', 'PIP_CONSTRAINT']
for var in vars_to_clear:
    if var in os.environ:
        del os.environ[var]
print('Environment variables cleared successfully.')

## Step 3: Install Core Linux Utilities & Build Libraries
Installs standard build tools and OpenJDK 17 required by modern Android SDK/NDK pipelines.

In [ ]:
!sudo apt update
!sudo apt install -y git zip unzip openjdk-17-jdk python3-pip autoconf automake autotools-dev libtool libtool-bin libltdl-dev pkg-config zlib1g-dev libncurses-dev libffi-dev libssl-dev cmake build-essential libsqlite3-dev sqlite3 bzip2 libbz2-dev libgdbm-dev libgdbm-compat-dev liblzma-dev libreadline-dev uuid-dev

## Step 4: Install Modern Buildozer & Cython
Installs standard, updated build tools.

In [ ]:
!pip install --upgrade pip
!pip install Cython buildozer kivymd virtualenv

## Step 5: Clone Project & Position Files
Clones the Open_E-Dope repository and shifts files to the working `/content/` root directory.

In [ ]:
!git clone https://github.com/Jgriffiths92/Open_E-Dope.git
!mv /content/Open_E-Dope/main.py  /content/
!mv /content/Open_E-Dope/buildozer.spec  /content/
!mv /content/Open_E-Dope/layout.kv  /content/
!mv /content/Open_E-Dope/extra_manifest.xml  /content/
!mv /content/Open_E-Dope/extra_intents.xml  /content/
!mv /content/Open_E-Dope/assets  /content/
!mv /content/Open_E-Dope/app  /content/
!mv /content/Open_E-Dope/res  /content/
!mv /content/Open_E-Dope/circularprogressbar.py  /content/

## Step 6: Compile Kivy App to APK
Executes a standard, un-wrapped build command. Remember to look at your `buildozer.spec` file in the sidebar to ensure dependencies (like `kivymd`) are typed into the `requirements` field before running.

In [ ]:
!set -o pipefail; buildozer -v android debug 2>&1 | tee /content/build.log

## Step 7: Download the Completed APK

In [ ]:
import os
from google.colab import files

bin_dir = '/content/bin'
if os.path.exists(bin_dir):
    apk_files = [f for f in os.listdir(bin_dir) if f.endswith('.apk')]
    if apk_files:
        for apk in apk_files:
            files.download(os.path.join(bin_dir, apk))
    else:
        print('No APK file found in the bin directory.')
else:
    print('Bin directory does not exist. Check your build log for errors.')